# NB09 v3 — Build04 Final Regulatory Investigation Agent

**Scientific AI Builder · Frozen Q1–Q10 official evaluation · final feature freeze**

This notebook derives from **NB09 v2** without modifying its original code, frozen corpus, Q1–Q10 question text, Build01–03 scores, v1/v2 logs, or retrieval configuration.

**v3 additions**: investigation-task coverage tracking, gap classification into local law / missing shipment facts / current-law verification / interpretation conflict / unsupported claim, critical issue-first searches, transparent stopping reasons, independent coverage metrics and safe unresolved-issues reporting.

**Experimental integrity**: Q4 and Q9 are development-exposed cases; Q1–Q3, Q5–Q8, Q10 are the final evaluation set for out-of-development comparison. Neither the human reference nor earlier results enter runtime planning/validation. All 10 are reported. Do not revise v3 after looking at final scores.

**Execution history**: the official protocol required offline tests with `RUN_OFFICIAL=False` and 50/50 retrieval parity before paid execution. Q1 could be used for operational debugging, followed by the remaining QIDs in the same frozen experiment; substantive scores were retained rather than repeatedly optimized. The completed v3 records are now frozen under `results/build04/`. Keep paid execution disabled for portfolio review. `ENABLE_CELLAR_LIVE=False` means no authoritative live article-level checking.


## 0. Installation and configuration

Install the shared dependencies from `requirements.txt`; `requirements-notebooks.txt` adds the optional Jupyter environment. The first code cell resolves the repository root from the kernel's working directory or its parents. For an external kernel, set `REGULATORY_AGENT_ROOT` to the absolute checkout path before setup.

SentenceTransformer model downloads, OpenAI requests and Cellar requests require network access. The repository's `tests/offline_smoke.py` provides selective offline validation without paid calls or changes to frozen outputs.


In [ ]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

# %pip install -U "langgraph>=0.2" "pydantic>=2" "openai>=1.60" "sentence-transformers" "rank-bm25" numpy pandas requests beautifulsoup4

In [1]:
import os, re, json, hashlib, platform, time, math
from datetime import datetime, timezone, date
from pathlib import Path
from getpass import getpass
from typing import TypedDict, Literal, NotRequired
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field, ValidationError
from langgraph.graph import StateGraph, START, END
from openai import OpenAI

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "Run NB09 v3 from Regulatory_agent repository root (data/ missing)."
MODEL_NAME = "gpt-6-luna"           # Same model family as Build03; use your connected API entitlement.
REASONING_EFFORT = "medium"
RUN_OFFICIAL = False               # PAID calls only if deliberately set True AFTER offline tests.
SELECT_QIDS = [f"Q{i}" for i in range(1, 11)]  # Freeze all Q1–Q10 for final run.
ENABLE_CELLAR_LIVE = False         # Metadata-only; not a current-law validity test.
MAX_INITIAL_SEARCHES = 5
MAX_TARGETED_SEARCHES = 3
MAX_TOTAL_SEARCHES = 8
MAX_VALIDATION_ROUNDS = 2
MAX_CELLAR_CALLS = 2
MAX_LLM_CALLS = 12                # Includes audit repair / re-audit.
MAX_OUTPUT_TOKENS = 2200
EXPERIMENT_ID = "NB09_build04_investigation_v3"
RUN_DIR = __import__("pathlib").Path("results/build04")  # Never overwrite v1/v2.
print("Python:", platform.python_version(), "| model:", MODEL_NAME)
print("Paid calls enabled:", RUN_OFFICIAL, "| live Cellar:", ENABLE_CELLAR_LIVE)
print("Experiment:", EXPERIMENT_ID, "| outputs:", RUN_DIR)


c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Python: 3.13.5 | model: gpt-6-luna
Paid calls enabled: False | live Cellar: False
Experiment: NB09_build04_investigation_v3 | outputs: data\build04_nb09_v3_runs


## 1. Load unchanged NB08/NB04b assets (frozen data gate)

The notebook reads the exact NB08 library and retrieval configuration; it does **not** regenerate or silently update them. The NB08 `benchmark_context` contains shared scenario information—not the evaluator's Verified Reference.

In [2]:
FILES = {
    "chunks": __import__("pathlib").Path("data/processed/chunks_legal_aware.json"),
    "benchmark": __import__("pathlib").Path("data/benchmark/benchmark_questions_v1.json"),
    "retrieval_config": __import__("pathlib").Path("data/processed/retrieval_config_nb03_final_hybrid_static_rag.json"),
    "nb03_results": __import__("pathlib").Path("data/processed/retrieval_results_nb03_final_hybrid_static_rag.csv"),
    "nb04b_config": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_experiment_config.json"),
    "nb04b_answers": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_answers_comparison.csv"),
}
for key, path in FILES.items():
    assert path.is_file(), f"Missing {key}: {path}"

legal_chunks = json.loads(FILES["chunks"].read_text(encoding="utf-8"))
benchmark = json.loads(FILES["benchmark"].read_text(encoding="utf-8"))
retrieval_config = json.loads(FILES["retrieval_config"].read_text(encoding="utf-8"))
nb04b_config = json.loads(FILES["nb04b_config"].read_text(encoding="utf-8"))
nb03_df = pd.read_csv(FILES["nb03_results"], encoding="utf-8-sig")
comparison_df = pd.read_csv(FILES["nb04b_answers"], encoding="utf-8-sig")
questions = benchmark["questions"]
benchmark_hash = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert len(legal_chunks) == 1383, "Library changed: investigate before running controlled test"
assert list(questions) == [f"Q{i}" for i in range(1,11)]
assert benchmark_hash == retrieval_config["benchmark_sha256"]
assert nb04b_config["model"] == MODEL_NAME
assert nb04b_config["reasoning_effort"] == REASONING_EFFORT
assert retrieval_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert retrieval_config["top_k"] == 5
assert retrieval_config["rrf_k"] == 60
assert retrieval_config["candidate_depth_per_retriever"] == 50
assert retrieval_config["diversification"]["max_per_group"] == 2
assert set(comparison_df.qid) == set(questions)

print("Benchmark:", benchmark["benchmark_id"])
print("Legal chunks:", len(legal_chunks), "| benchmark questions:", len(questions))
print("Frozen NB04b answers available:", comparison_df.qid.nunique())

Benchmark: regulatory_agent_v1_notion_2026-10-08
Legal chunks: 1383 | benchmark questions: 10
Frozen NB04b answers available: 10


## 2. Reuse NB08's exact hybrid search and ordered parity gate

Embedding computation can take time on the first run. The **50/50 gate must pass** before any official Build04 evaluation. During official runs, `hybrid_search` uses the identical five-law corpus and retrieval mechanics as NB03/NB08.

In [3]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [safe(chunk.get(k)) for k in (
        "regulation_title", "file", "location", "article", "heading", "text"
    )]
    return " | ".join(p for p in parts if p).strip()

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

legal_texts = [searchable_text(ch) for ch in legal_chunks]
assert all(legal_texts)
EMBEDDING_MODEL = retrieval_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
legal_embeddings = embedding_model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
bm25 = BM25Okapi([tokenize(t) for t in legal_texts])
print("NB03-equivalent retriever components ready:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

NB03-equivalent retriever components ready: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
RRF_K = int(retrieval_config["rrf_k"])
CANDIDATE_DEPTH = int(retrieval_config["candidate_depth_per_retriever"])
MAX_PER_SOURCE_GROUP = int(retrieval_config["diversification"]["max_per_group"])

def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))
    return (file_name, article if article else location)

def hybrid_search(query: str, top_k: int = 5):
    query = str(query).strip()
    if not query:
        raise ValueError("query must be non-empty")
    if not (1 <= int(top_k) <= 10):
        raise ValueError("top_k must be in 1..10")
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]
    fused_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    fused_order = sorted(fused_scores, key=fused_scores.get, reverse=True)
    selected, counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = source_group(chunk)
        if counts.get(group, 0) >= MAX_PER_SOURCE_GROUP:
            continue
        selected.append((idx, chunk, fused_scores[idx]))
        counts[group] = counts.get(group, 0) + 1
        if len(selected) == int(top_k):
            break
    results = []
    for rank, (idx, chunk, rrf_score) in enumerate(selected, start=1):
        results.append({
            "rank": rank, "chunk_index": int(idx),
            "chunk_id": chunk.get("chunk_id"), "rrf_score": float(rrf_score),
            "file": chunk.get("file"), "regulation_title": chunk.get("regulation_title"),
            "article": chunk.get("article"), "location": chunk.get("location"),
            "heading": chunk.get("heading"), "text": chunk.get("text"),
        })
    return results

In [5]:
PARITY_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build04_nb09_v3_retrieval_parity_audit.csv")
audit_rows, parity_by_qid = [], {}
for qid, question in questions.items():
    baseline = nb03_df[nb03_df.qid == qid].sort_values("rank")
    expected = baseline.chunk_id.astype(str).tolist()
    actual = [r["chunk_id"] for r in hybrid_search(question, top_k=5)]
    assert len(expected) == len(actual) == 5
    parity_by_qid[qid] = expected == actual
    for rank, (a,b) in enumerate(zip(expected,actual), start=1):
        audit_rows.append({"qid":qid,"rank":rank,"nb03_chunk_id":a,"nb09_chunk_id":b,"match":a==b})
    print(f"{qid}: {'PASS' if parity_by_qid[qid] else 'MISMATCH'}")
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(PARITY_PATH, index=False, encoding="utf-8-sig")
matched = int(audit_df.match.sum())
PARITY_OK = matched == 50 and all(parity_by_qid.values())
print(f"\nPARITY: {matched}/50 ordered chunks | Saved: {PARITY_PATH}")
assert PARITY_OK, "STOP: original NB03 retrieval parity failed. No official NB09 run allowed."

Q1: PASS
Q2: PASS
Q3: PASS
Q4: PASS
Q5: PASS
Q6: PASS
Q7: PASS
Q8: PASS
Q9: PASS
Q10: PASS

PARITY: 50/50 ordered chunks | Saved: data\build04_nb09_v3_retrieval_parity_audit.csv


## 3. Reuse NB08 Cellar metadata checker unchanged

This is the **existing** `retrieve_cellar_current_law` implementation, including CELEX allowlisting, official EU Publications Office SPARQL, failure-closed parsing and flags `article_level_changes_verified=False`, `current_legal_status_verified=False`. The returned date is only a **latest-observed consolidation candidate**. The web request is **not made** until an enabled workflow asks for it.

In [6]:
LAW_REGISTRY = {
    "2023/915": {"celex":"32023R0915", "eli":"reg/2023/915", "snapshot_date":"2026-08-19", "snapshot_file":"EU_2023_915_consolidated_2026-08-19.html", "title":"Contaminants maximum levels"},
    "2019/1793": {"celex":"32019R1793", "eli":"reg_impl/2019/1793", "snapshot_date":"2026-06-30", "snapshot_file":"EU_2019_1793_consolidated_2026-06-30.html", "title":"Increased import controls / special conditions"},
    "2017/625": {"celex":"32017R0625", "eli":"reg/2017/625", "snapshot_date":"2025-01-05", "snapshot_file":"EU_2017_625_consolidated_2025-01-05.html", "title":"Official controls"},
    "178/2002": {"celex":"32002R0178", "eli":"reg/2002/178", "snapshot_date":"2026-01-01", "snapshot_file":"EU_2002_178_consolidated_2026-01-01.html", "title":"General Food Law"},
    "2019/1013": {"celex":"32019R1013", "eli":"reg_impl/2019/1013", "snapshot_date":None, "snapshot_file":"EU_2019__1013_original.html", "title":"Prior-notification arrangements"},
}

for key, item in LAW_REGISTRY.items():
    assert (__import__("pathlib").Path("data/legal_sources") / item["snapshot_file"]).is_file(), f"Snapshot missing for {key}"
    item["official_basic_url"] = f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A{item["celex"]}'

assert set(LAW_REGISTRY) == {"2023/915","2019/1793","2017/625","178/2002","2019/1013"}
print("Current-law checker scope:", list(LAW_REGISTRY))

Current-law checker scope: ['2023/915', '2019/1793', '2017/625', '178/2002', '2019/1013']


In [7]:
# Official Cellar SPARQL: CELEX identifiers are stored under
# cdm:resource_legal_id_celex (NOT cdm:work_id_document).
# No EUR-Lex HTML scraping, no user-supplied URLs, no fabricated latest dates.
from urllib.parse import urlparse

CELLAR_SPARQL = "https://publications.europa.eu/webapi/rdf/sparql"
CELLAR_CELEX_BASE = "https://publications.europa.eu/resource/celex/"
CELLAR_TIMEOUT = (10, 45)
MAX_SPARQL_BYTES = 2_000_000
MAX_VERSIONS = 100


def _base_record(regulation):
    law = LAW_REGISTRY.get(regulation, {})
    return {
        "regulation": regulation, "basic_celex": law.get("celex"),
        "snapshot_date": law.get("snapshot_date"),
        "snapshot_file": law.get("snapshot_file"),
        "official_basic_url": law.get("official_basic_url"),
        "official_metadata_endpoint": CELLAR_SPARQL,
        "checked_utc": datetime.now(timezone.utc).isoformat(),
        "status": "unverified", "source": "EU Publications Office / Cellar SPARQL",
        "latest_date_candidate": None, "latest_celex_candidate": None,
        "observed_consolidations": [], "date_comparison": "not_verified",
        "article_level_changes_verified": False,
        "application_date_verified": False,
        "current_legal_status_verified": False,
        "complete_consolidation_list_verified": False,
    }


def _version_celex_pattern(basic_celex):
    # Sector 3 basic act -> sector 0 consolidated versions.
    return re.compile(r"^0" + re.escape(basic_celex[1:]) + r"-(\d{8})$")


def _sparql_versions_query(celex, mode="relation"):
    assert re.fullmatch(r"3\d{4}R\d{4}", celex), "Only allowlisted regulation CELEX IDs"
    prefix = "0" + celex[1:] + "-"
    header = "PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>\n"
    if mode == "relation":
        # Both documented relation predicates; incoming edges point at base work.
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?original cdm:resource_legal_id_celex "{celex}" .
  {{ ?consolidated cdm:act_consolidated_based_on_resource_legal ?original . }}
  UNION
  {{ ?consolidated cdm:act_consolidated_consolidates_resource_legal ?original . }}
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(STRSTARTS(STR(?consolidated_celex), "{prefix}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    if mode == "celex_index":
        # Validated against live Cellar: the plain SPARQL CONTAINS filter
        # returned 19 CELEX records (including 02019R1793-20260630).
        # The old Virtuoso bif:contains clause silently returned zero rows.
        # Search broadly, then use _parse_versions() to accept ONLY the
        # correct sector-0 consolidated CELEX IDs with valid dates.
        search_fragment = celex[1:]  # e.g. 2019R1793
        return header + f"""SELECT DISTINCT ?consolidated ?consolidated_celex WHERE {{
  ?consolidated cdm:resource_legal_id_celex ?consolidated_celex .
  FILTER(CONTAINS(STR(?consolidated_celex), "{search_fragment}"))
}} ORDER BY DESC(?consolidated_celex) LIMIT {MAX_VERSIONS}"""
    raise ValueError("Unknown SPARQL query mode")


def _run_cellar_sparql(query, session=None):
    client = session if session is not None else requests.Session()
    response = client.get(
        CELLAR_SPARQL, params={"query": query},
        headers={"Accept": "application/sparql-results+json",
                 "User-Agent": "ScientificAIBuilder-Research/0.2"},
        timeout=CELLAR_TIMEOUT, allow_redirects=True,
    )
    resolved = getattr(response, "url", CELLAR_SPARQL)
    parsed = urlparse(resolved)
    if parsed.scheme != "https" or parsed.hostname != "publications.europa.eu":
        raise ValueError("untrusted_redirect")
    if response.status_code != 200:
        raise ValueError(f"http_{response.status_code}")
    raw = response.content
    if not raw or len(raw) > MAX_SPARQL_BYTES:
        raise ValueError("empty_or_oversize_response")
    # Virtuoso can return HTTP 200 with plain-text query error.
    content_type = response.headers.get("Content-Type", "").lower()
    if "json" not in content_type:
        raise ValueError("non_json_or_sparql_error")
    data = response.json()
    if not isinstance(data.get("results", {}).get("bindings"), list):
        raise ValueError("invalid_sparql_result_structure")
    return data["results"]["bindings"]


def _parse_versions(bindings, basic_celex, as_of=None):
    as_of = as_of or datetime.now(timezone.utc).date()
    pattern = _version_celex_pattern(basic_celex)
    found = {}
    for row in bindings:
        value = row.get("consolidated_celex", {}).get("value", "")
        match = pattern.fullmatch(value)
        if not match:
            continue
        try:
            dt = datetime.strptime(match.group(1), "%Y%m%d").date()
        except ValueError:
            continue
        # Future-dated consolidations are recorded separately, not current.
        if dt > as_of:
            continue
        found[value] = {
            "celex": value,
            "date": dt.isoformat(),
            "official_celex_url": CELLAR_CELEX_BASE + value,
        }
    return sorted(found.values(), key=lambda v: (v["date"],v["celex"]), reverse=True)


def retrieve_cellar_current_law(regulation, session=None, as_of=None):
    """Official metadata check only; NEVER certify legal force or amended clauses.

    Relational lookup + independently scoped CELEX-index lookup.
    Missing results or failed confirmation are UNVERIFIED, not 'no update'.
    """
    rec = _base_record(regulation)
    if regulation not in LAW_REGISTRY:
        rec["error"] = "not_allowlisted"
        return rec
    basic = LAW_REGISTRY[regulation]["celex"]
    as_of = as_of or datetime.now(timezone.utc).date()
    if isinstance(as_of, str):
        as_of = date.fromisoformat(as_of)
    rec["as_of_date"] = as_of.isoformat()

    try:
        rel = _run_cellar_sparql(_sparql_versions_query(basic,"relation"),session)
        rec["relation_row_count"] = len(rel)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        rel = []
        rec["relation_error"] = f"{type(exc).__name__}: {exc}"

    # Important: index query is independent, not assumed equivalent to links.
    try:
        idx = _run_cellar_sparql(_sparql_versions_query(basic,"celex_index"),session)
        rec["celex_index_row_count"] = len(idx)
    except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
        idx = []
        rec["celex_index_error"] = f"{type(exc).__name__}: {exc}"

    observed = _parse_versions(rel+idx,basic,as_of=as_of)
    rec["observed_consolidations"] = observed
    rec["lookup_methods"] = ["relation", "celex_index"]
    if not observed:
        rec["error"] = "no_verified_consolidation_rows"
        return rec

    candidate = observed[0]
    rec["latest_date_candidate"] = candidate["date"]
    rec["latest_celex_candidate"] = candidate["celex"]
    rec["latest_candidate_official_url"] = candidate["official_celex_url"]

    # If either query failed, retrieval may be incomplete.
    if "relation_error" in rec or "celex_index_error" in rec:
        rec["status"] = "partial_metadata"
        rec["date_comparison"] = "latest_observed_only_incomplete_verification"
        return rec
    if len(rel) >= MAX_VERSIONS or len(idx) >= MAX_VERSIONS:
        rec["status"] = "partial_metadata"
        rec["error"] = "result_limit_reached"
        return rec

    rec["status"] = "metadata_found"
    rec["complete_consolidation_list_verified"] = False  # Graph freshness/coverage not provable here.
    snapshot = rec["snapshot_date"]
    if snapshot is None:
        rec["date_comparison"] = "original_snapshot_without_consolidation_date"
    elif candidate["date"] > snapshot:
        rec["date_comparison"] = "newer_consolidation_observed"
    elif candidate["date"] == snapshot:
        rec["date_comparison"] = "matches_latest_observed_consolidation"
    else:
        rec["date_comparison"] = "latest_observed_older_than_snapshot_review"
    return rec

# Alias preserves existing tool wiring below.
retrieve_eurlex_basic_page = retrieve_cellar_current_law

## 4. Strict exact provision retrieval and law-scoped search (v2)

`get_provision` accepts only one **specific** Article (`Article 10`), numbered row (`row 1.1.3`), or Annex + numbered logical row (`Annex II, Section 1, row 14`). A bare `Annex II` is accepted for human exploration, but **never auto-selected as an exact targeted retry**, because it spans many provisions. Composite prose like `Annex II and provisions governing entry ...` is **not** a valid exact identifier and is routed to focused search. A parsed amending-document Annex is never treated as a different Act's Annex.

`hybrid_search_scoped` uses the same dense embeddings and BM25 objects as NB03, but ranks **within one explicitly allowed law** for targeted follow-up. It does not change `hybrid_search` or the 50/50 baseline parity.


In [8]:
ALLOWED_REGULATIONS = frozenset(LAW_REGISTRY)
FILE_TO_LAW = {v["snapshot_file"]: k for k,v in LAW_REGISTRY.items()}

_ARTICLE = re.compile(r"^(?:Article|Art\.?)\s+(\d+)(?:\s*\(\d+\))?$",re.I)
_ANNEX = re.compile(
    r"^Annex\s+(IIa|IV|III|II|I)(?:\s*,?\s*Section\s+(\d+))?"
    r"(?:\s*,?\s*row\s+(\d+(?:\.\d+)*))?$", re.I)
_ROW = re.compile(r"^row\s+(\d+(?:\.\d+)*)$",re.I)

def parse_provision(provision: str) -> dict:
    """Reject vague/multi-unit strings; do not infer a target from one substring."""
    p = " ".join(str(provision).split()).strip(" ,")
    if not p or len(p)>100: raise ValueError("Provide one exact Article/Annex/row (<=100 chars)")
    m=_ARTICLE.fullmatch(p)
    if m:return {"kind":"article","article":f"Article {int(m.group(1))}"}
    m=_ANNEX.fullmatch(p)
    if m:
        kind="annex_row" if m.group(3) else "annex_broad"
        return {"kind":kind,"annex":m.group(1).upper(),"section":m.group(2),"row":m.group(3)}
    m=_ROW.fullmatch(p)
    if m:return {"kind":"row","row":m.group(1)}
    raise ValueError("Ambiguous exact provision; use targeted search or a single 'Article N' / 'Annex II, Section 1, row 14'")

def is_precise_provision(provision: str) -> bool:
    try:return parse_provision(provision)["kind"] in {"article","row","annex_row"}
    except ValueError:return False

def _numbered_row_match(text, row):
    # Explicit row label or synthetic legal-entry chunk only, not incidental number in prose.
    return bool(re.search(r"(?:^|\bRow:|\blogical row)\s*"+re.escape(row)+r"(?=\s*(?:[—|,;.]|$))",text,re.I))

def get_provision(regulation_id: str, provision: str, top_k: int=5) -> list[dict]:
    """Strict local retrieval from a named regulation/document version. No legal applicability inference."""
    if regulation_id not in ALLOWED_REGULATIONS:
        raise ValueError(f"Regulation not allowlisted: {regulation_id}")
    spec=parse_provision(provision)
    file_name=LAW_REGISTRY[regulation_id]["snapshot_file"]
    n=max(1,min(int(top_k),10))
    candidates=[]
    for ch in legal_chunks:
        if ch.get("file")!=file_name:continue
        loc=str(ch.get("location") or "")
        txt=str(ch.get("text") or "")
        art=str(ch.get("article") or "")
        kind=spec["kind"]
        if kind=="article": matched=(art.casefold()==spec["article"].casefold())
        else:
            annex=spec.get("annex")
            annex_match=(not annex or bool(re.search(r"\bAnnex\s+"+re.escape(annex)+r"\b",loc,re.I)))
            if kind=="annex_broad":
                matched=annex_match and bool(re.search(r"\bAnnex\s+"+re.escape(annex)+r"\b",loc,re.I))
            else:
                row=spec["row"]
                location_match=bool(re.search(r"\brow\s+"+re.escape(row)+r"\b",loc,re.I))
                text_match=_numbered_row_match(txt,row)
                matched=annex_match and (location_match or text_match)
                if spec.get("section"):
                    # The row-group reconstruction records section in location; fail closed otherwise.
                    matched=matched and bool(re.search(r"\bSection\s+"+re.escape(spec["section"])+r"\b",loc,re.I))
        if matched:candidates.append(ch)
    candidates.sort(key=lambda c:(0 if "logical row" in str(c.get("location","")).lower() else 1,str(c.get("chunk_id",""))))
    return [dict(c) for c in candidates[:n]]

def hybrid_search_scoped(query: str, regulation_id: str, top_k: int=5) -> list[dict]:
    """Dense + BM25 + RRF over ONE frozen regulation; NB03 global retrieval remains untouched."""
    if regulation_id not in ALLOWED_REGULATIONS:raise ValueError("Regulation is not allowlisted")
    q=str(query).strip()
    if not q:raise ValueError("Focused query required")
    target_file=LAW_REGISTRY[regulation_id]["snapshot_file"]
    ids=np.asarray([i for i,ch in enumerate(legal_chunks) if ch.get("file")==target_file],dtype=int)
    if not len(ids):return []
    q_emb=embedding_model.encode([q],convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False)[0]
    ds=legal_embeddings[ids]@q_emb
    bs=np.asarray(bm25.get_scores(tokenize(q)),dtype=float)[ids]
    dense_order=ids[np.argsort(ds)[::-1][:CANDIDATE_DEPTH]]
    bm25_order=ids[np.argsort(bs)[::-1][:CANDIDATE_DEPTH]]
    fused={}
    for order in (dense_order,bm25_order):
        for rank,idx in enumerate(order,start=1):
            idx=int(idx);fused[idx]=fused.get(idx,0)+1.0/(RRF_K+rank)
    selected=[];groups={}
    for idx in sorted(fused,key=fused.get,reverse=True):
        ch=legal_chunks[idx];key=source_group(ch)
        if groups.get(key,0)>=MAX_PER_SOURCE_GROUP:continue
        groups[key]=groups.get(key,0)+1
        selected.append({"rank":len(selected)+1,"chunk_index":idx,"chunk_id":ch.get("chunk_id"),
            "rrf_score":float(fused[idx]),"file":ch.get("file"),"regulation_title":ch.get("regulation_title"),
            "article":ch.get("article"),"location":ch.get("location"),"heading":ch.get("heading"),"text":ch.get("text")})
        if len(selected)>=max(1,min(int(top_k),10)):break
    return selected

def make_evidence(chunk:dict,retrieved_by:str)->dict:
    file_name=str(chunk.get("file") or "")
    law_id=FILE_TO_LAW.get(file_name)
    if not law_id:raise ValueError(f"Unknown frozen file: {file_name}")
    law=LAW_REGISTRY[law_id]
    chunk_id=str(chunk.get("chunk_id") or "")
    evidence_id="E"+hashlib.sha256(chunk_id.encode()).hexdigest()[:10]
    date_part=("-"+law["snapshot_date"].replace("-", "")) if law.get("snapshot_date") else ""
    consolidated_celex=("0"+law["celex"][1:]+date_part) if date_part else law["celex"]
    return {"evidence_id":evidence_id,"chunk_id":chunk_id,"regulation_id":law_id,
       "source_document_celex":consolidated_celex,"basic_celex":law["celex"],"source_file":file_name,
       "source_version":law.get("snapshot_date") or "original_snapshot",
       "article":chunk.get("article"),"location":chunk.get("location"),
       "source_url":f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:{consolidated_celex}',
       "text":str(chunk.get("text") or ""),"retrieved_by":retrieved_by}

print("Strict get_provision + law-scoped hybrid search ready | laws:",len(ALLOWED_REGULATIONS))


Strict get_provision + law-scoped hybrid search ready | laws: 5


## 5. Agent State, structured outputs and execution controls

Node functions return **partial state updates**; no mutable global Tool history is shared between different QIDs. Local provenance records are frozen by `evidence_id` (derived from `chunk_id`). Benchmark references and evaluation scores are **not in this state**.

In [9]:
class InvestigationTask(BaseModel):
    task_id: str
    question: str
    priority: Literal["high","medium","low"] = "medium"
    search_query: str = Field(min_length=5)
    expected_evidence_type: str = "relevant legal provisions"
    needs_currentness: bool = False

class PlanOutput(BaseModel):
    tasks: list[InvestigationTask] = Field(min_length=1, max_length=7)

class SuggestedGap(BaseModel):
    description: str
    query: str
    regulation_id: str | None = None
    exact_provision: str | None = None
    critical: bool = True

class LegalClaim(BaseModel):
    statement: str
    evidence_ids: list[str] = Field(default_factory=list)
    support_quote: str = ""
    status: Literal["supported", "unsupported", "conflicting", "unverified", "not_applicable"] = "unverified"
    scope_note: str = ""

class ValidationOutput(BaseModel):
    claims: list[LegalClaim] = Field(default_factory=list)
    gaps: list[SuggestedGap] = Field(default_factory=list)
    conflicts: list[str] = Field(default_factory=list)

class CoverageItem(BaseModel):
    # A sub-issue may refine (not delete) the frozen planner tasks; all parent tasks remain auditable.
    coverage_id: str
    task_id: str
    issue: str
    status: Literal["pending","covered","partially_covered","missing_legal_evidence",
                    "missing_case_fact","requires_live_verification","unresolved_conflict","not_applicable"]
    gap_type: Literal["none","legal_evidence","case_fact","current_law", "interpretation_conflict", "unsupported_claim"] = "none"
    critical: bool = True
    evidence_ids: list[str] = Field(default_factory=list)
    claim_indices: list[int] = Field(default_factory=list)
    missing_elements: list[str] = Field(default_factory=list)
    regulation_id: str | None = None
    exact_provision: str | None = None
    search_query: str = ""
    rationale: str = ""

class CoverageOutput(BaseModel):
    items: list[CoverageItem] = Field(default_factory=list)

class AuditOutput(BaseModel):
    issues: list[str] = Field(default_factory=list)
    pass_audit: bool = False
    revised_report: str = ""

class AgentState(TypedDict, total=False):
    qid: str
    question: str
    case_context: dict
    investigation_plan: list[dict]
    research_queue: list[dict]
    evidence: list[dict]
    claims: list[dict]
    gaps: list[dict]
    conflicts: list[str]
    current_law_checks: list[dict]
    source_facts: list[dict]
    coverage_items: list[dict]
    coverage_summary: dict
    classified_gaps: list[dict]
    audit_resolution: str
    original_draft: str
    search_calls: int
    initial_searches: int
    targeted_searches: int
    cellar_calls: int
    validation_rounds: int
    llm_calls: int
    usage: dict
    searched_signatures: list[str]
    trace: list[dict]
    stop_reason: str
    draft_answer: str
    final_answer: str
    audit_issues: list[str]


def new_state(question: str, qid: str = "USER", case_context: dict | None = None) -> AgentState:
    return AgentState(
        qid=qid,question=question,case_context=case_context or {},
        investigation_plan=[], research_queue=[], evidence=[],claims=[],gaps=[],
        conflicts=[], current_law_checks=[],source_facts=[],coverage_items=[],coverage_summary={},classified_gaps=[],audit_resolution="",original_draft="",search_calls=0,initial_searches=0,
        targeted_searches=0,cellar_calls=0,validation_rounds=0,llm_calls=0,
        usage={"input_tokens":0,"output_tokens":0}, searched_signatures=[],
        trace=[], stop_reason="",draft_answer="",final_answer="",audit_issues=[])

# Explicit run safety checks.
assert MAX_TOTAL_SEARCHES == MAX_INITIAL_SEARCHES + MAX_TARGETED_SEARCHES
assert MAX_VALIDATION_ROUNDS >= 1
assert MAX_LLM_CALLS >= 4

## 6. OpenAI LLM interface — API only when `RUN_OFFICIAL=True`

The API uses the **Responses API** pattern from NB04b (`reasoning={"effort":"medium"}`, `store=False`) to preserve model configuration. Structured responses are parsed and validated locally with Pydantic. If JSON extraction fails, the workflow **fails clearly**, rather than silently inventing a plan or validated claims.

In [10]:
CLIENT = None
SHARED_CONTEXT = nb04b_config["benchmark_context"]
SHARED_SYSTEM_PROMPT = nb04b_config["system_prompt"]
# No eval references or question-specific must-have answers enter this context.

GLOBAL_RULES = """You are assisting with EU food-law research using a frozen five-law corpus.
Never invent legal limits, section numbers, provisions, obligations or applicability.
Distinguish a regulation's own Annex from the Annex of a different amending act.
A consolidating document date / Cellar metadata is NOT proof of effective law.
Always distinguish confirmed document text, shipment facts and legal applicability.
A single chemical detection is NOT automatically a legal exceedance.
Do not claim current article-level status unless genuinely verified.
Only support a legal claim using retrieved legal text and explicit evidence IDs.
No benchmark references, evaluation rubrics or earlier model grades are accessible."""

LLM_USAGE = {"input_tokens":0,"output_tokens":0,"calls":0}

def extract_json_object(text: str) -> dict:
    s=text.strip()
    if s.startswith("```"):
        s=re.sub(r"^```(?:json)?\s*", "", s,flags=re.I)
        s=re.sub(r"\s*```$", "", s)
    try: return json.loads(s)
    except json.JSONDecodeError:
        decoder=json.JSONDecoder()
        for i,c in enumerate(s):
            if c=="{":
                try:
                    obj,offset=decoder.raw_decode(s[i:])
                    if isinstance(obj,dict):return obj
                except json.JSONDecodeError:pass
        raise ValueError("LLM output was not valid JSON object")

def llm_call(task_instructions: str, payload: dict, *, json_schema=None, max_output_tokens=MAX_OUTPUT_TOKENS):
    global CLIENT
    if not RUN_OFFICIAL:
        raise RuntimeError("LLM disabled; set RUN_OFFICIAL=True deliberately")
    if CLIENT is None:
        if not os.getenv("OPENAI_API_KEY"):
            os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (hidden): ").strip()
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY missing")
        CLIENT = OpenAI()
    if LLM_USAGE["calls"]>=MAX_LLM_CALLS:
        raise RuntimeError("LLM call budget exhausted")
    response=CLIENT.responses.create(
        model=MODEL_NAME,reasoning={"effort":REASONING_EFFORT},
        instructions="\n\n".join([SHARED_CONTEXT,SHARED_SYSTEM_PROMPT,GLOBAL_RULES,task_instructions]),
        input=json.dumps(payload,ensure_ascii=False),
        max_output_tokens=max_output_tokens,store=False,
    )
    LLM_USAGE["calls"]+=1
    usage=getattr(response,"usage",None)
    LLM_USAGE["input_tokens"] += int(getattr(usage,"input_tokens",0) or 0)
    LLM_USAGE["output_tokens"] += int(getattr(usage,"output_tokens",0) or 0)
    output_text=getattr(response,"output_text",None)
    if not output_text:
        output_text="\n".join(str(getattr(part,"text","")) for block in getattr(response,"output",[]) for part in getattr(block,"content",[]))
    if not output_text.strip():
        raise RuntimeError("No model output; consider increasing max_output_tokens for reasoning")
    if json_schema is None:
        return output_text.strip()
    data=extract_json_object(output_text)
    return json_schema.model_validate(data)

def compact_sources(evidence:list[dict], max_items=24, text_chars=1350):
    # Ensure new evidence is represented; older evidence is retained in full in State.
    return [{k:e.get(k) for k in ("evidence_id","regulation_id","source_document_celex", "article","location","source_version","source_url")} |
           {"text":e["text"][:text_chars]} for e in evidence[-max_items:]]

## 7. Planner — question-specific *research intentions*, not prefilled answers

At most seven tasks are planned. Search queries may mention a regulation, but the planner must **not** treat its guesses as legal proof. The first retrieval wave is bounded to five calls.

In [11]:
def plan_investigation(state:AgentState)->dict:
    instructions="""Act ONLY as an EU regulatory INVESTIGATION PLANNER. Return a JSON object:
    {"tasks":[{"task_id":"T1","question":"...","priority":"high",
    "search_query":"...","expected_evidence_type":"...","needs_currentness":false}]}.
    Select 1 to 7 concrete research tasks tailored to the user's question.
    Cover important legal issues without reproducing answers, numeric limits or unverifiable assertions.
    Prefer high-value searches; max FIVE will be run initially. Do not use an evaluation checklist.
    Broad cross-cutting questions should be decomposed into several distinct legal issue areas,
    including applicable requirements, responsible actors, procedures and conditional responses
    ONLY insofar as they are relevant to the user question. Do not provide legal answers.
    The available corpus is a frozen snapshot. Do not ask the local searcher to prove
    today's current legal effectiveness. Instead separate snapshot law from currentness
    gaps for later official external verification."""
    output=llm_call(instructions,{"question":state["question"],"case_context":state["case_context"]},json_schema=PlanOutput)
    tasks=[t.model_dump() for t in output.tasks]
    order={"high":0,"medium":1,"low":2}
    tasks.sort(key=lambda x:order[x["priority"]])
    queries=[{"mode":"search","query":t["search_query"],"task_id":t["task_id"],"phase":"initial","top_k":5} for t in tasks[:MAX_INITIAL_SEARCHES]]
    return {"investigation_plan":tasks,"research_queue":queries,"llm_calls":LLM_USAGE["calls"],
            "trace":state["trace"]+[{"node":"planner","planned_tasks":len(tasks),"queued":len(queries)}]}

## 8. Retriever — strict exact queries plus law-scoped targeted retrieval

Initial queries use exactly the frozen NB03 global hybrid search. In targeted phases, ambiguous `exact_provision` suggestions are deliberately **downgraded to law-scoped search**, not interpreted as whatever Annex appears first. Exact results that return only previously seen chunks may make **one law-scoped fallback within the same search call**. Each operation is logged with the number of novel Evidence IDs. Live Cellar is still metadata-only.


In [12]:
def search_signature(item:dict)->str:
    return json.dumps({k:item.get(k) for k in ("mode","query","regulation_id","provision")},sort_keys=True).casefold()

def retrieve_evidence(state:AgentState)->dict:
    by_id={e["evidence_id"]:e for e in state.get("evidence",[])}
    seen=set(state.get("searched_signatures",[]))
    trace=list(state["trace"])
    initial=state["initial_searches"];targeted=state["targeted_searches"]
    total=state["search_calls"];cellar=state["cellar_calls"]
    currentness=list(state["current_law_checks"])
    current_laws_seen={c.get("regulation") for c in currentness}
    for item in state.get("research_queue",[]):
        sig=search_signature(item)
        if sig in seen:
            trace.append({"node":"retriever","skipped_duplicate":sig});continue
        phase=item.get("phase","initial")
        if total>=MAX_TOTAL_SEARCHES:
            trace.append({"node":"retriever","stop":"total_search_budget"});break
        if phase=="initial" and initial>=MAX_INITIAL_SEARCHES:break
        if phase=="targeted" and targeted>=MAX_TARGETED_SEARCHES:break
        seen.add(sig)
        law=item.get("regulation_id")
        mode=item.get("mode","search")
        actual_mode=mode;fallback_used=False;error=None
        try:
            if mode=="exact":
                if law not in ALLOWED_REGULATIONS or not is_precise_provision(item.get("provision","")):
                    raise ValueError("Malformed exact query (must specify one precise provision)")
                chunks=get_provision(law,item["provision"],top_k=item.get("top_k",5))
                # Avoid another v1 failure: an exact hit with zero NEW evidence is not progress.
                if not chunks or all(make_evidence(c,"exact")["evidence_id"] in by_id for c in chunks):
                    chunks=hybrid_search_scoped(item.get("query") or item["provision"],law,top_k=item.get("top_k",5))
                    fallback_used=True;actual_mode="scoped_search_after_stale_exact"
            elif mode=="search" and law in ALLOWED_REGULATIONS:
                chunks=hybrid_search_scoped(item["query"],law,top_k=item.get("top_k",5))
                actual_mode="scoped_search"
            elif mode=="search":
                chunks=hybrid_search(item["query"],top_k=item.get("top_k",5))
            else:raise ValueError("Unsupported retrieval mode")
        except Exception as exc:
            error=f"{type(exc).__name__}: {exc}"
            try:
                if law in ALLOWED_REGULATIONS:
                    chunks=hybrid_search_scoped(item.get("query") or "EU food law",law,top_k=item.get("top_k",5))
                    fallback_used=True;actual_mode="scoped_search_after_error"
                else:chunks=[]
            except Exception as exc2:
                chunks=[];error+=f" / fallback={type(exc2).__name__}: {exc2}"
        total+=1
        if phase=="initial":initial+=1
        else:targeted+=1
        added=[]
        for ch in chunks:
            e=make_evidence(ch,retrieved_by=actual_mode)
            if e["evidence_id"] not in by_id:
                by_id[e["evidence_id"]]=e;added.append(e["evidence_id"])
        trace.append({"node":"retriever","requested_mode":mode,"actual_mode":actual_mode,
              "phase":phase,"query":item.get("query") or item.get("provision"),
              "provision":item.get("provision"),"law_scope":law,
              "retrieved":len(chunks),"added_ids":added,"new_evidence_count":len(added),
              "fallback_used":fallback_used,"error":error,"search_count":total})
    # Optional; metadata alone never marks provisions legally applicable today.
    if ENABLE_CELLAR_LIVE and cellar<MAX_CELLAR_CALLS:
        for task in state.get("investigation_plan",[]):
            if not task.get("needs_currentness") or cellar>=MAX_CELLAR_CALLS:continue
            planned_laws=re.findall(r"\b(?:2023/915|2019/1793|2017/625|178/2002|2019/1013)\b",task["search_query"])
            if not planned_laws:planned_laws=list(dict.fromkeys(e["regulation_id"] for e in by_id.values()))
            for law in planned_laws:
                if law not in current_laws_seen and cellar<MAX_CELLAR_CALLS:
                    try:rec=retrieve_cellar_current_law(law)
                    except Exception as exc:rec={"regulation":law,"status":"unverified","error":str(exc)}
                    currentness.append(rec);current_laws_seen.add(law);cellar+=1
                    trace.append({"node":"retriever","tool":"check_current_law","regulation":law,
                                  "status":rec.get("status"),"article_level_verified":False})
    return {"evidence":list(by_id.values()),"research_queue":[],"searched_signatures":sorted(seen),
            "search_calls":total,"initial_searches":initial,"targeted_searches":targeted,
            "cellar_calls":cellar,"current_law_checks":currentness,"trace":trace}


## 9. Validator v2 — deterministic evidence gate + source-fact coverage + semantic legal reasoning

The quote/CELEX checks remain deterministic. A new **generic source-fact ledger** extracts explicitly written product scope, named hazards, representative source CN code, and inspection frequency from applicable logical-row chunks. These are immutable **snapshot observations**, not legal conclusions about a consignment in 2026. Source facts are candidates for completeness, *never benchmark-derived expected answers*. They remain linked to literal quotes and are persisted across validation rounds.

LLM validation can reject applicability or identify missing operative Articles. Unsupported narrative claims may **not** be promoted to `supported` without a verified quote. The validator keeps earlier supported claims; if contradictory evidence later appears, human legal review is required.


In [13]:
def normalized_ws(s:str)->str:
    return " ".join(str(s).casefold().split())

def deterministic_validate_claim(claim:dict,evidence_map:dict[str,dict])->dict:
    c=dict(claim)
    ids=list(dict.fromkeys(c.get("evidence_ids",[])))
    valid=[i for i in ids if i in evidence_map]
    missing=[i for i in ids if i not in evidence_map]
    c["evidence_ids"]=valid
    quote=normalized_ws(c.get("support_quote",""))
    quote_found=bool(len(quote)>=12 and any(quote in normalized_ws(evidence_map[i]["text"]) for i in valid))
    c["quote_verified"]=quote_found
    c["invalid_evidence_ids"]=missing
    if c.get("status")=="supported" and (not valid or not quote_found):
        c["status"]="unverified"
        c["scope_note"]=(c.get("scope_note","")+" | Quote/evidence-ID check failed").strip()
    c["current_applicability_status"]="unverified"
    return c

def extract_source_facts(evidence:list[dict],question:str,case_context:dict|None=None)->list[dict]:
    """Extract *literal* structured row facts; no benchmark answers or generated values."""
    q=normalized_ws(question+" "+json.dumps(case_context or {},ensure_ascii=False))
    result=[]
    for e in evidence:
        t=e.get("text","");loc=e.get("location","") or ""
        # This convention is produced by the original NB02 row-group fixer.
        # No hardcoded commodity, country, Annex row, numerical limit or hazard name.
        if "Product scope:" not in t or "Hazards listed for this logical entry:" not in t:
            continue
        if "logical row" not in normalized_ws(loc+" "+t):continue
        # Guard against another state's country/product entry being mistaken for the case.
        origin=re.search(r"logical row\s+\d+\s+—\s+([^()\n.]+)\(([A-Z]{2})\)",t,re.I)
        if origin and normalized_ws(origin.group(1)).strip() not in q:
            continue
        pattern={
            "product_scope":r"(Product scope:\s*[^.\n]+\.)",
            "hazards":r"(Hazards listed for this logical entry:\s*[^.\n]+\.)",
            "inspection_frequency":r"(Frequency of identity and physical checks:\s*\d+(?:[.,]\d+)?%)",
            "commodity_code_example":r"(HTML row \d+:\s*\d+\s*\|\s*[^|]+\|\s*[^|]+\|\s*(?:ex\s+)?\d{4}\s+\d{2}\s+\d{2})"
        }
        # Full natural phrase from source must be present in question/case;
        # avoid pushing an unrelated country's row into a response.
        product=t.split("Product scope:",1)[1].split("Hazards listed",1)[0].casefold()
        # Conservative lexical anchor. For a different product, this adapts from question text.
        tokens=[w for w in re.findall(r"[a-zà-ÿ]{4,}",normalized_ws(product)) if w not in {"including","dried","product","prepared","containing","mixtures","paste","preserved"}]
        if tokens and not any(w in q for w in tokens[:6]):continue
        for category,p in pattern.items():
            match=re.search(p,t,re.I)
            if not match:continue
            quote=match.group(1).rstrip(" ")
            if len(quote)<12:continue
            fid="F"+hashlib.sha256((e["evidence_id"]+category+quote).encode()).hexdigest()[:10]
            result.append({"fact_id":fid,"category":category,"evidence_id":e["evidence_id"],
                      "regulation_id":e["regulation_id"],"source_version":e["source_version"],
                      "source_location":loc,"quote":quote,
                      "snapshot_claim":f'Frozen Regulation (EU) {e["regulation_id"]} at {loc}: {quote}'})
    return list({x["fact_id"]:x for x in result}.values())

def source_fact_claim(fact:dict,evidence_map:dict)->dict:
    """Grounded snapshot observation; never inferred as proof of today's applicability."""
    return deterministic_validate_claim({"statement":fact["snapshot_claim"],
       "evidence_ids":[fact["evidence_id"]],"support_quote":fact["quote"],
       "status":"supported","scope_note":"Direct source quotation; snapshot only, legal applicability unverified",
       "origin":"deterministic_source_fact"},evidence_map)

def merge_validated_claims(*collections:list[dict])->list[dict]:
    out=[];seen=set()
    for collection in collections:
        for c in collection:
            if c.get("status")!="supported" or not c.get("quote_verified"):continue
            key=(normalized_ws(c.get("statement","")),tuple(sorted(c.get("evidence_ids",[]))))
            if key not in seen:out.append(c);seen.add(key)
    return out

def validate_evidence(state:AgentState)->dict:
    source_facts=extract_source_facts(state["evidence"],state["question"],state.get("case_context"))
    instructions="""You are an EU regulatory EVIDENCE VALIDATOR. Return JSON:
    {"claims":[{"statement":"...","evidence_ids":["E..."],
    "support_quote":"literal >=12 character quote in the cited source",
    "status":"supported|unsupported|conflicting|unverified|not_applicable",
    "scope_note":"..."}],
    "gaps":[{"description":"...","query":"focused actionable search phrase",
    "regulation_id":null,"exact_provision":null,"critical":true}],"conflicts":["..."]}.
    The supplied SOURCE_FACTS are literal observations from retrieved chunks (not benchmark answers).
    COVER all material source facts relevant to the user question, including hazards and control frequencies;
    do not merely mention an Annex without considering the legal Article(s) that give it effect.
    Do NOT claim official current applicability merely because the law text is consolidated.
    Never conflate the Annex of an amending act with the Annex of the amended act.
    If the Annex listing is found but its operative requirements are missing, search for the
    implementing Articles; do not assume it has no regulatory effect.
    Always distinguish a snapshot observation, a supported operative legal requirement, and a
    case-specific applicability fact (e.g. CN/TARIC classification). Do not invent provisions.
    Prefer one precise missing Article if it is explicitly cited by evidence; otherwise request a
    narrow search within the relevant regulation. Keep <=12 claims, <=5 actionable gaps.
    Never use earlier answers, grading rubrics or benchmark reference notes."""
    payload={"question":state["question"],"case_context":state.get("case_context",{}),
       "plan":state["investigation_plan"],"evidence":compact_sources(state["evidence"]),
       "source_facts":source_facts,
       "prior_supported_claims":[c for c in state.get("claims",[]) if c.get("status")=="supported"],
       "current_law_metadata":[{k:x.get(k) for k in ("regulation","status","latest_date_candidate",
             "article_level_changes_verified","application_date_verified")} for x in state["current_law_checks"]]}
    output=llm_call(instructions,payload,json_schema=ValidationOutput,max_output_tokens=3800)
    evidence_map={e["evidence_id"]:e for e in state["evidence"]}
    model_claims=[deterministic_validate_claim(c.model_dump(),evidence_map) for c in output.claims[:12]]
    direct_claims=[source_fact_claim(f,evidence_map) for f in source_facts]
    # Source-fact claims are always framed as 'Frozen source says X'; no status escalation.
    supported=merge_validated_claims(state.get("claims",[]),model_claims,direct_claims)
    unresolved=[c for c in model_claims if c.get("status")!="supported"]
    claims=supported+unresolved
    gaps=[g.model_dump() for g in output.gaps[:5]]
    # Do not fabricate a search query by copying a legal conclusion verbatim.
    rounds=state["validation_rounds"]+1
    covered=[f["fact_id"] for f in source_facts if any(c.get("origin")=="deterministic_source_fact"
            and f["evidence_id"] in c.get("evidence_ids",[]) and f["quote"]==c.get("support_quote") for c in claims)]
    return {"claims":claims,"source_facts":source_facts,"gaps":gaps,"conflicts":output.conflicts,
       "validation_rounds":rounds,"llm_calls":LLM_USAGE["calls"],
       "trace":state["trace"]+[{"node":"validator","round":rounds,"claims":len(claims),
               "supported":len(supported),"gaps":len(gaps),"conflicts":len(output.conflicts),
               "extracted_source_facts":len(source_facts),"covered_source_facts":len(covered)}]}


## 9B. Coverage Tracker and Gap Classification — v3

The planner's tasks are fixed before retrieval. After each validation round, a separate coverage check assesses each necessary legal issue, not simply whether an existing claim has a quote. Supported evidence IDs are verified again deterministically. Missing shipment facts and today's legal status are **non-searchable** within the frozen corpus. A coverage item can be subdivided for a broad synthesis question, but cannot insert the human evaluation must-have checklist.


In [14]:
# 9B. Coverage Tracking (v3): separate issue completeness from literal claim support.
# This assessment is built from the user's question and the pre-retrieval investigation plan,
# not from benchmark answers or historical human grades.
COVERAGE_GAP_TYPES={"none","legal_evidence","case_fact","current_law","interpretation_conflict","unsupported_claim"}
COVERAGE_STATUSES={"pending","covered","partially_covered","missing_legal_evidence","missing_case_fact","requires_live_verification","unresolved_conflict","not_applicable"}


def normalize_coverage_item(item:dict, state:AgentState)->dict:
    """Conservative rule-based gate; semantic judgment is NEVER treated as citation proof."""
    item=dict(item)
    plan={t["task_id"]:t for t in state["investigation_plan"]}
    if item.get("task_id") not in plan:
        raise ValueError("Coverage item references a non-planned parent task")
    evidence={e["evidence_id"] for e in state.get("evidence",[])}
    supported=validated_claims(state)
    good_ids={eid for c in supported for eid in c.get("evidence_ids",[])}
    # Retain only evidence backed by deterministic quote-verified claims.
    item["evidence_ids"]=[eid for eid in dict.fromkeys(item.get("evidence_ids",[])) if eid in evidence and eid in good_ids]
    valid_idx=[]
    for k in item.get("claim_indices",[]):
        if isinstance(k,int) and 0<=k<len(supported) and k not in valid_idx:
            valid_idx.append(k)
    item["claim_indices"]=valid_idx
    # A claimed 'covered' issue without supported sources is not covered.
    if item["status"]=="covered" and not item["evidence_ids"]:
        item["status"]="missing_legal_evidence" if item.get("gap_type") in ("none","legal_evidence") else "partially_covered"
        item["missing_elements"]=[*item.get("missing_elements",[]),"No quote-verified linked source"]
    # Do not look for real-world facts or current live law inside the static corpus.
    forced={"missing_case_fact":"case_fact","requires_live_verification":"current_law",
            "unresolved_conflict":"interpretation_conflict","missing_legal_evidence":"legal_evidence"}
    if item["status"] in forced:item["gap_type"]=forced[item["status"]]
    if item["status"] in ("covered","not_applicable"):item["gap_type"]="none"
    if item["status"]=="partially_covered" and item.get("gap_type")=="none":item["gap_type"]="legal_evidence"
    if item["status"]=="pending" and item.get("gap_type")=="none":item["gap_type"]="legal_evidence"
    # Require a meaningful query before a legal gap consumes retrieval budget.
    item["search_query"]=str(item.get("search_query") or "").strip()
    item["regulation_id"]=item.get("regulation_id") if item.get("regulation_id") in ALLOWED_REGULATIONS else None
    if not is_precise_provision(item.get("exact_provision") or ""):
        item["exact_provision"]=None
    return item


def summarize_coverage(items:list[dict], state:AgentState)->dict:
    # Parent tasks with no assessment remain 'pending'; cannot silently vanish.
    parent_ids={x.get("task_id") for x in items}
    missing=[t["task_id"] for t in state["investigation_plan"] if t["task_id"] not in parent_ids]
    critical=[x for x in items if x.get("critical",True) and x.get("status")!="not_applicable"]
    covered=[x for x in critical if x["status"]=="covered"]
    unresolved=[x for x in critical if x["status"]!="covered"]
    by_type={k:sum(1 for x in items if x.get("gap_type")==k and x.get("status") not in ("covered","not_applicable"))
             for k in sorted(COVERAGE_GAP_TYPES) if k!="none"}
    return {"items":len(items),"critical_covered":len(covered),"critical_total":len(critical),
        "critical_coverage_ratio":round(len(covered)/len(critical),4) if critical else None,
        "unresolved_critical":len(unresolved),"missing_parent_tasks":missing,
        "unresolved_by_type":by_type,
        "note":"LLM-derived diagnostic coverage; NOT benchmark completeness or legal correctness."}


def assess_investigation_coverage(state:AgentState)->dict:
    supported=validated_claims(state)
    instr="""Assess COVERAGE of an EU legal investigation, not whether existing claims quote sources.
    Use ONLY the ORIGINAL USER QUESTION, case context, the PRE-RETRIEVAL PLAN and retrieved
    verified claims. Return CoverageOutput JSON {"items":[...]}, 1–15 distinct issues.
    Every planned task MUST have at least one coverage item; you may split a broad task into
    several narrower issues when the user's question clearly requires multiple legal
    conclusions. Preserve its original task_id; use distinct coverage_id C1, C2, etc.
    DO NOT copy a benchmark answer, hidden rubric, specific Q-number, or earlier system result.
    Inspect coverage at the issue/conclusion level, not the source-fact-count level.
    'covered' only if evidence exists AND the precise issue is resolved by a verified claim;
    attach its verified evidence_ids and (optional) zero-based claim_indices.
    A source being retrieved is not sufficient if the relevant conclusion is unaddressed.
    Be skeptical of broad task wording; explicitly flag unresolved distinct legal subissues.
    Use statuses: covered, partially_covered, missing_legal_evidence, missing_case_fact,
    requires_live_verification, unresolved_conflict, not_applicable, pending.
    Classify gaps as: none, legal_evidence, case_fact, current_law,
    interpretation_conflict or unsupported_claim.
    Missing real shipment concentrations, documents or customs classification = case_fact;
    current legal effect / amendment date check = current_law. NEVER search frozen corpus
    for missing case facts or to certify today's law.
    When requesting missing legal provisions, make search_query precise and regulation_id
    explicit if actually known; exact_provision must be ONE unambiguous Article or Annex row,
    never a vague list. Distinguish the Annex of a different amending act.
    Keep critical=True only for issues necessary to answer the user's question. Avoid
    speculative requirements unrelated to the user's request.
    No hardcoded legal references are provided to you by the coverage checker."""
    payload={"question":state["question"],"case_context":state.get("case_context",{}),
      "pre_retrieval_plan":state["investigation_plan"],
      "supported_claims":[{**{k:c.get(k) for k in ("statement","evidence_ids","support_quote","scope_note")},"claim_index":i}
                          for i,c in enumerate(supported)],
      "unresolved_validator_gaps":state.get("gaps",[]),
      "evidence_catalog":[{k:e.get(k) for k in ("evidence_id","regulation_id","article","location","source_version")}
                          for e in state.get("evidence",[])],
      "previous_coverage":state.get("coverage_items",[])}
    result=llm_call(instr,payload,json_schema=CoverageOutput,max_output_tokens=4000)
    seen=set(); items=[]
    for it in result.items[:15]:
        x=normalize_coverage_item(it.model_dump(),state)
        k=x["coverage_id"]
        if k in seen:continue
        seen.add(k);items.append(x)
    # Do not let the LLM omit parent tasks from the assessment.
    for t in state["investigation_plan"]:
        if not any(x["task_id"]==t["task_id"] for x in items):
            items.append(normalize_coverage_item({"coverage_id":"UNASSESSED_"+t["task_id"],
                "task_id":t["task_id"],"issue":t["question"],"status":"pending",
                "gap_type":"legal_evidence","critical":t.get("priority")!="low",
                "evidence_ids":[],"claim_indices":[],"missing_elements":["Coverage not assessed"],
                "search_query":t["search_query"],"rationale":"Planner task cannot disappear"},state))
    summary=summarize_coverage(items,state)
    classified=[{k:x.get(k) for k in ("coverage_id","task_id","issue","gap_type","status",
         "critical","search_query","regulation_id","exact_provision","missing_elements")}
        for x in items if x["status"] not in ("covered","not_applicable")]
    return {"coverage_items":items,"coverage_summary":summary,"classified_gaps":classified,
        "llm_calls":LLM_USAGE["calls"],
        "trace":state["trace"]+[{"node":"coverage_tracker","round":state["validation_rounds"],
            "critical_covered":summary["critical_covered"],"critical_total":summary["critical_total"],
            "unresolved_by_type":summary["unresolved_by_type"]}]}


## 10. Stopping rules and targeted re-search v2

A `gap.exact_provision` is **only** an exact lookup when the **entire value** is a single Article or numbered row. `Annex II and provisions governing entry` must become **law-scoped hybrid search**, not an accidental Annex-only lookup. A question asking for today's effective law cannot be solved by re-reading a historical snapshot: such a live-only gap is explicitly **deferred** when Cellar live lookup is disabled.


In [15]:
def _requires_external_currentness(gap:dict)->bool:
    """True for questions that cannot be certified by any frozen text alone."""
    s=normalized_ws(gap.get("description","")+" "+gap.get("query",""))
    return (not ENABLE_CELLAR_LIVE and bool(re.search(
       r"(?:currently effective|latest applicable|as-of-date|on the shipment date|live current|"
       r"actual amendments? and effective dates|current legal applicability)",s)))

def build_targeted_queue(state:AgentState)->list[dict]:
    seen=set(state.get("searched_signatures",[]))
    slots=max(0,min(MAX_TARGETED_SEARCHES-state["targeted_searches"],MAX_TOTAL_SEARCHES-state["search_calls"]))
    priority={"interpretation_conflict":0,"legal_evidence":1,"unsupported_claim":2}
    problems=[]
    if state.get("coverage_items"):
        for x in state["coverage_items"]:
            if x.get("status") in ("covered","not_applicable"):continue
            if not x.get("critical",True):continue
            kind=x.get("gap_type")
            if kind not in priority:continue  # No futile searches for case facts / live status.
            problems.append({"description":x.get("issue",""),"query":x.get("search_query", ""),
                "regulation_id":x.get("regulation_id"),"exact_provision":x.get("exact_provision"),
                "critical":True,"gap_type":kind,"coverage_id":x.get("coverage_id")})
    else:  # Compatibility with v2 offline tests / older incomplete states.
        for gap in state.get("gaps",[]):
            if not gap.get("critical",True) or _requires_external_currentness(gap):continue
            problems.append({**gap,"gap_type":"legal_evidence"})
    problems.sort(key=lambda g:(priority.get(g.get("gap_type"),9), bool(not g.get("exact_provision"))))
    out=[]
    for gap in problems:
        law=gap.get("regulation_id")
        if law not in ALLOWED_REGULATIONS:
            matches=[rid for rid in ALLOWED_REGULATIONS if rid in str(gap.get("query", ""))]
            law=matches[0] if len(matches)==1 else None
        provision=str(gap.get("exact_provision") or "").strip()
        query=str(gap.get("query") or "").strip()
        if len(query)<5:continue
        exact=law in ALLOWED_REGULATIONS and is_precise_provision(provision)
        item={"phase":"targeted","mode":"exact" if exact else "search","query":query,"top_k":5}
        if law in ALLOWED_REGULATIONS:item["regulation_id"]=law
        if exact:item["provision"]=provision
        if gap.get("coverage_id"):item["coverage_id"]=gap["coverage_id"]
        sig=search_signature(item)
        if sig in seen:continue
        seen.add(sig);out.append(item)
        if len(out)>=slots:break
    return out


def decide_next_step(state:AgentState)->Literal["targeted_research","report","report_with_gaps"]:
    supported=validated_claims(state)
    if state.get("coverage_items"):
        unfinished=[x for x in state["coverage_items"] if x.get("critical",True)
                    and x["status"] not in ("covered","not_applicable")]
        if not unfinished and supported and not state.get("conflicts"):return "report"
    else:
        critical=[g for g in state.get("gaps",[]) if g.get("critical",True)]
        if not critical and not state.get("conflicts") and supported:return "report"
    if state["validation_rounds"]>=MAX_VALIDATION_ROUNDS:return "report_with_gaps"
    if state["targeted_searches"]>=MAX_TARGETED_SEARCHES or state["search_calls"]>=MAX_TOTAL_SEARCHES:
        return "report_with_gaps"
    return "targeted_research" if build_targeted_queue(state) else "report_with_gaps"

def plan_research_gap(state:AgentState)->dict:
    queue=build_targeted_queue(state)
    deferred=[x.get("issue", "") for x in state.get("coverage_items",[]) if x.get("gap_type") in ("current_law","case_fact")]
    return {"research_queue":queue,"trace":state["trace"]+[
        {"node":"gap_planner","targeted_queries":queue,"deferred_live_only":deferred}]}


## 11. Reporter + repair-first Final Audit v2

Reports can only quote **validated claim IDs**. If the first semantic audit fails, the auditor's revision is *not* accepted blindly: it must pass an Evidence-ID gate and a second independent semantic audit. Otherwise a **structured snapshot-only fallback** retains every verified claim/fact, not just the two short findings left in Q4 v1. Debugging issues are preserved in JSON audit logs, not dumped as sprawling report prose.


In [16]:
def validated_claims(state:AgentState)->list[dict]:
    return [c for c in state.get("claims",[]) if c.get("status")=="supported" and c.get("quote_verified")]

def supported_evidence_ids(state:AgentState)->set[str]:
    return {eid for c in validated_claims(state) for eid in c.get("evidence_ids",[])}

def citation_audit(report:str,state:AgentState)->list[str]:
    """Reject uncited or invented evidence IDs, without treating semantic correctness as proven."""
    cited=set(re.findall(r"\[(E[0-9a-f]{10})\]",report))
    any_tag=set(re.findall(r"\[(E[^\]\s]+)\]",report))
    valid=supported_evidence_ids(state)
    invalid=sorted(cited-valid)
    errors=[]
    if invalid:errors.append("Citations not attached to supported claims: "+", ".join(invalid))
    if any_tag-cited:errors.append("Malformed evidence IDs: "+", ".join(sorted(any_tag-cited)))
    if validated_claims(state) and not cited:errors.append("Supported findings must carry evidence-ID citations")
    return errors

def safe_fallback_report(state:AgentState,issues:list[str]|None=None)->str:
    """Safe, CONTENT-PRESERVING snapshot report; never claim today's legal status."""
    supported=validated_claims(state)
    lines=["## Regulatory findings — verified against frozen document text"]
    if supported:
        for c in supported:
            cites=" ".join(f"[{eid}]" for eid in c["evidence_ids"])
            lines.append("- "+c["statement"].strip()+" "+cites)
    else:lines.append("- No legally substantive conclusion was adequately supported by retrieved evidence.")
    lines.extend(["","## Important questions not yet resolved"])
    unique=set()
    for g in state.get("gaps",[]):
        desc=str(g.get("description") or "").strip()
        if desc and desc.casefold() not in unique:
            lines.append("- "+desc);unique.add(desc.casefold())
    for x in state.get("coverage_items",[]):
        if x.get("critical",True) and x.get("status") not in ("covered","not_applicable"):
            desc="Coverage item unresolved ("+x.get("gap_type","unknown")+"): "+x.get("issue","")
            if desc.casefold() not in unique:lines.append("- "+desc);unique.add(desc.casefold())
    for c in state.get("claims",[]):
        if c.get("status") in ("unsupported","unverified","conflicting"):
            desc="Additional evidence required for: "+c.get("statement","")
            if desc.casefold() not in unique:lines.append("- "+desc);unique.add(desc.casefold())
    if not unique:lines.append("- Confirm product/CN classification, shipment status and law in force on the relevant date.")
    lines.extend(["","## Important distinction", "- The claims above describe text found in a fixed, dated corpus, not proof that it applies today or to this particular consignment.",
       "- A reported detection is not by itself proof that a numerical legal limit was exceeded. Further action depends on evidence, product scope and shipment stage.",
       "- The report is an evidence-limited research note, not a binding clearance or enforcement decision."])
    return "\n".join(lines)

def draft_report(state:AgentState)->dict:
    supported=validated_claims(state)
    instruction="""Write a concise, actionable EU food-operator regulatory research memo with:
    1. Short assessment; 2. Evidence-supported law; 3. Facts/currentness to verify;
    4. Practical next checks.
    Report any CRITICAL uncovered investigation issues separately, giving their gap type.
    Do not state that an investigation is complete when critical issues remain unresolved.
    Do not present missing shipment facts or current-law verification gaps as legal searches.
    No legal statement may go beyond the supplied VALIDATED CLAIMS. The evidence quotes
    from the same legal row should be retained in the memo when relevant, including named
    hazards, control frequencies and product scope. Cite only provided [EvidenceID].
    Do not cite retrieved chunks that have no validated claim. Don't assert that a law is
    currently applicable based only on a snapshot or Cellar date.
    Distinguish factual detection from non-compliance and precautionary advice from mandatory
    detention, refusal or recall. Say 'not established' where evidence is missing.
    Never invent an Article/Annex, amount, rate or operative requirement."""
    if not supported:
        draft=safe_fallback_report(state)
    else:
        draft=llm_call(instruction,{"question":state["question"],"case_context":state.get("case_context",{}),
                  "validated_claims":supported,"source_facts":state.get("source_facts",[]),
                  "gaps":state["gaps"],"coverage_items":state.get("coverage_items",[]),
                  "coverage_summary":state.get("coverage_summary",{}),"conflicts":state["conflicts"],
                  "sources":compact_sources([e for e in state["evidence"] if e["evidence_id"] in supported_evidence_ids(state)],
                                            max_items=18,text_chars=1200)},max_output_tokens=3100)
    return {"draft_answer":draft,"original_draft":draft,"llm_calls":LLM_USAGE["calls"],
            "trace":state["trace"]+[{"node":"reporter","length":len(draft),"supported":len(supported)}]}

_AUDIT_PROMPT="""Audit an EU legal research memo using ONLY the supplied supported claims and literal quotations.
 Return JSON {"issues":["..."],"pass_audit":true|false,"revised_report":"..."}.
 Mark false for any material assertion of an unverified Article obligation, incorrect source scope,
 fake current legal status, or confusion between detection and exceedance.
 If false, propose a revised memo retaining ALL safely supported facts and removing/qualifying
 unsupported claims. Use ONLY listed [EvidenceID] citations and preserve the currentness caveat.
 Also audit whether the memo conceals a critical unfinished COVERAGE issue; it must
 transparently state uncovered items and not invent their answers.
 If the draft is already sound, pass_audit=true and revised_report may be empty."""

def audit_report(state:AgentState)->dict:
    issues=citation_audit(state["draft_answer"],state)
    final=None;resolution=""
    allowed=[c for c in validated_claims(state)]
    if not allowed:
        final=safe_fallback_report(state,issues);resolution="no_supported_claims"
    elif not issues:
        try:
            first=llm_call(_AUDIT_PROMPT,{"draft":state["draft_answer"],
                "supported_claims":allowed,"gaps":state["gaps"],"coverage_items":state.get("coverage_items",[]),"question":state["question"]},
                json_schema=AuditOutput,max_output_tokens=3000)
            issues.extend(first.issues)
            if first.pass_audit and not first.issues:
                final=state["draft_answer"];resolution="original_passed"
            else:
                proposal=(first.revised_report or "").strip()
                deterministic_issues=citation_audit(proposal,state) if proposal else ["Auditor did not supply a revision"]
                if not deterministic_issues and LLM_USAGE["calls"]<MAX_LLM_CALLS:
                    try:
                        second=llm_call(_AUDIT_PROMPT,{"draft":proposal,
                           "supported_claims":allowed,"gaps":state["gaps"],"coverage_items":state.get("coverage_items",[]),"question":state["question"],
                           "original_issues":issues},json_schema=AuditOutput,max_output_tokens=2500)
                        if second.pass_audit and not second.issues and not citation_audit(proposal,state):
                            final=proposal;resolution="revised_and_reaudited"
                        else:issues.extend(["Re-audit: "+x for x in second.issues])
                    except Exception as exc:
                        issues.append("Re-audit error: "+type(exc).__name__+": "+str(exc)[:180])
                else:issues.extend(deterministic_issues)
        except Exception as exc:
            issues.append("Semantic audit error: "+type(exc).__name__+": "+str(exc)[:180])
    if final is None:
        final=safe_fallback_report(state,issues)
        resolution="structured_supported_claims_fallback"
    return {"final_answer":final,"audit_issues":issues,"audit_resolution":resolution,
         "llm_calls":LLM_USAGE["calls"],
         "usage":{"input_tokens":LLM_USAGE["input_tokens"],"output_tokens":LLM_USAGE["output_tokens"]},
         "trace":state["trace"]+[{"node":"auditor","issues":issues,
                       "resolution":resolution,"used_fallback":resolution.endswith("fallback")} ]}


## 12. Compile the LangGraph Investigation Agent

A conditional edge enforces at most two validation passes. LangGraph manages state transitions, while the Retriever and Stopping Rules enforce the actual budgets.

In [17]:
workflow=StateGraph(AgentState)
workflow.add_node("planner",plan_investigation)
workflow.add_node("retriever",retrieve_evidence)
workflow.add_node("validator",validate_evidence)
workflow.add_node("coverage_tracker",assess_investigation_coverage)
workflow.add_node("gap_planner",plan_research_gap)
workflow.add_node("reporter",draft_report)
workflow.add_node("auditor",audit_report)
workflow.add_edge(START,"planner")
workflow.add_edge("planner","retriever")
workflow.add_edge("retriever","validator")
workflow.add_edge("validator","coverage_tracker")
workflow.add_conditional_edges("coverage_tracker",decide_next_step,{
    "targeted_research":"gap_planner",
    "report":"reporter",
    "report_with_gaps":"reporter",
})
workflow.add_edge("gap_planner","retriever")
workflow.add_edge("reporter","auditor")
workflow.add_edge("auditor",END)
agent_graph=workflow.compile()
print("Build04 graph compiled: planner → retriever → validator → coverage_tracker ↔ gap_planner → reporter → auditor")

Build04 graph compiled: planner → retriever → validator → coverage_tracker ↔ gap_planner → reporter → auditor


## 13. Offline regression suite — **no API calls**

In addition to the v1 gates, v2 tests specifically reproduce Q4's three failure modes: vague `Annex II and provisions` cannot be an exact call, the law-scoped search can seek missing operative Articles, hazard, CN code and frequency **already in the retrieved row** are preserved as evidence-backed facts, and a fallback preserves every supported finding without truncated content. The tests do **not** use any frozen answers or evaluation checklist as input to the Agent.


In [18]:
if not RUN_OFFICIAL:
    article=get_provision("2019/1793","Article 11")
    assert article and all(x.get("article")=="Article 11" for x in article)
    figs=get_provision("2023/915","row 1.1.3")
    assert figs and any("Row: 1.1.3" in x.get("text","") for x in figs)
    imported=get_provision("2019/1793","Annex II, Section 1, row 14")
    assert imported and any("annexII_row14_turkiye_figs" in x.get("chunk_id","") for x in imported)
    assert get_provision("2019/1793","Article 9999")==[]
    for vague in ("Annex II and provisions governing entry", "Annex II, Section 1, Türkiye dried-fig entry", "Articles 9–11"):
        assert not is_precise_provision(vague),vague
        try:get_provision("2019/1793",vague);raise AssertionError("Vague provision was accepted")
        except ValueError:pass
    assert not is_precise_provision("Annex II")  # broad Annex is a SEARCH task in the agent
    assert is_precise_provision("Article 10")
    assert is_precise_provision("Annex II row 14")
    try:get_provision("2025/1441","Annex I");raise AssertionError("non-corpus Act allowed")
    except ValueError:pass

    row_e=make_evidence(imported[0],"exact")
    found=extract_source_facts([row_e],questions["Q4"],{"benchmark_case":"Türkiye-origin dried figs"})
    cats={f["category"] for f in found}
    assert {"product_scope","hazards","inspection_frequency","commodity_code_example"}<=cats,(cats,found)
    assert any("ochratoxin A" in f["quote"] for f in found if f["category"]=="hazards")
    assert any("30%" in f["quote"] for f in found if f["category"]=="inspection_frequency")
    assert any("0804 20 90" in f["quote"] for f in found if f["category"]=="commodity_code_example")
    evidence_map={row_e["evidence_id"]:row_e}
    supported=[source_fact_claim(f,evidence_map) for f in found]
    assert len(supported)>=3 and all(c["status"]=="supported" and c["quote_verified"] for c in supported)
    assert all(c["current_applicability_status"]=="unverified" for c in supported)
    assert "ochratoxin A" in safe_fallback_report({"claims":supported,"gaps":[]})
    assert "30%" in safe_fallback_report({"claims":supported,"gaps":[]})

    # User's observed v1 failure: three vague purported exact requests yielded 0 new evidence.
    q4_gap={"description":"Missing legal conditions","query":"EU 2019/1793 Annex II entry official certificate analytical report",
        "regulation_id":"2019/1793","exact_provision":"Annex II and provisions governing entry of Annex II consignments","critical":True}
    q4_state=new_state(questions["Q4"],"Q4")
    q4_state.update({"gaps":[q4_gap],"evidence":[row_e],"searched_signatures":[]})
    queue=build_targeted_queue(q4_state)
    assert len(queue)==1 and queue[0]["mode"]=="search" and queue[0]["regulation_id"]=="2019/1793",queue
    assert any(x.get("article")=="Article 10" for x in get_provision("2019/1793","Article 10"))

    sample=make_evidence(figs[0],"exact")
    valid={"statement":"Snapshot text confirmed","evidence_ids":[sample["evidence_id"]],
           "support_quote":sample["text"][20:80],"status":"supported","scope_note":""}
    assert deterministic_validate_claim(valid,{sample["evidence_id"]:sample})["quote_verified"]
    assert deterministic_validate_claim(dict(valid,support_quote="claim absent from this document"),
                                        {sample["evidence_id"]:sample})["status"]=="unverified"
    assert deterministic_validate_claim(dict(valid,evidence_ids=["Enotfound"]),
                                        {sample["evidence_id"]:sample})["status"]=="unverified"
    assert not citation_audit(safe_fallback_report({"claims":supported,"gaps":[]}),{"claims":supported})
    assert citation_audit("Invalid [E0000000000]",{"claims":supported})

    ss=new_state("Synthetic stop test")
    ss.update({"validation_rounds":MAX_VALIDATION_ROUNDS,"gaps":[{"description":"missing","query":"missing legal source","critical":True}]})
    assert decide_next_step(ss)=="report_with_gaps"
    ss2=new_state("Synthetic stop test")
    ss2.update({"gaps":[q4_gap]})
    assert decide_next_step(ss2)=="targeted_research"
    ss3=new_state("Synthetic stop test")
    ss3.update({"claims":supported,"gaps":[],"conflicts":[]})
    assert decide_next_step(ss3)=="report"
    print("PASS: v2 exact parser; row 14; source facts; vague-gap scoped routing; citation audit; safe fallback; stop rules")
else:
    print("SKIP offline tests in paid mode — run them first with RUN_OFFICIAL=False")


PASS: v2 exact parser; row 14; source facts; vague-gap scoped routing; citation audit; safe fallback; stop rules


## 13B. v3 offline coverage regression (no API)

These synthetic tests cover quote-backed evidence requirements, false `covered` classification, excluding missing case facts and live-currentness questions from static law searches, task completeness and hard stop rules. They do not give the model any Q9 answer keys.


In [19]:
# v3 TESTS: deterministic coverage gate and type-aware queue. No API/network/model calls.
if not RUN_OFFICIAL:
    tstate=new_state("What provisions govern this example?", "SYNTHETIC")
    tstate["investigation_plan"]=[{"task_id":"T1","question":"Find legal basis","priority":"high","search_query":"food law article", "expected_evidence_type":"law"}]
    tstate["evidence"]=[row_e]
    tstate["claims"]=[source_fact_claim(f,evidence_map) for f in found]
    covered=normalize_coverage_item({"coverage_id":"C1","task_id":"T1","issue":"Documented Annex entry",
        "status":"covered","gap_type":"none","evidence_ids":[row_e["evidence_id"]],"claim_indices":[],
        "critical":True,"missing_elements":[],"search_query":""},tstate)
    assert covered["status"]=="covered"
    unsupported=normalize_coverage_item({"coverage_id":"C2","task_id":"T1","issue":"Undocumented obligation",
        "status":"covered","gap_type":"none","evidence_ids":["E0000000000"],"claim_indices":[],
        "critical":True,"missing_elements":[],"search_query":"Article missing"},tstate)
    assert unsupported["status"]=="missing_legal_evidence"
    tstate["coverage_items"]=[covered,unsupported,
      normalize_coverage_item({"coverage_id":"C3","task_id":"T1","issue":"Missing concentration",
       "status":"missing_case_fact","gap_type":"case_fact","critical":True,"evidence_ids":[],
       "claim_indices":[],"missing_elements":["Measured sample concentration"],
       "search_query":"Look up measured actual shipment aflatoxin result"},tstate),
      normalize_coverage_item({"coverage_id":"C4","task_id":"T1","issue":"Current version unknown",
       "status":"requires_live_verification","gap_type":"current_law","critical":True,
       "evidence_ids":[],"claim_indices":[],"missing_elements":[],"search_query":"Find today's law online"},tstate)]
    q=build_targeted_queue(tstate)
    assert len(q)==1 and "Article missing" in q[0]["query"],q
    assert summarize_coverage(tstate["coverage_items"],tstate)["critical_coverage_ratio"]==0.25
    assert decide_next_step(tstate)=="targeted_research"
    tstate["validation_rounds"]=MAX_VALIDATION_ROUNDS
    assert decide_next_step(tstate)=="report_with_gaps"
    assert "Missing concentration" in safe_fallback_report(tstate)
    # Parent task cannot be marked complete merely by retrieving one unrelated source fact.
    tstate["coverage_items"]=[unsupported]
    assert summarize_coverage(tstate["coverage_items"],tstate)["critical_covered"]==0
    print("PASS: v3 coverage validation, missing-fact/live gap exclusion, critical queue, stop gate and report preservation")
else:
    print("SKIP v3 offline tests during official mode")


PASS: v3 coverage validation, missing-fact/live gap exclusion, critical queue, stop gate and report preservation


In [20]:
RUN_OFFICIAL = False
SELECT_QIDS = [f"Q{i}" for i in range(1, 11)]
ENABLE_CELLAR_LIVE = False

## 14. Official run — OFF by default; frozen Q1–Q10 records

**This section requires OpenAI API usage when enabled.** The official v3 evaluation is complete; its `Q#.json` records and manifest are stored under `results/build04/`. Keep `RUN_OFFICIAL=False` for portfolio review. The run-control cell immediately before this section sets the flag again, so a change made only in setup does not determine the final run configuration.

The historical protocol required the offline tests and 50/50 retrieval parity to pass before an authorized run. Q1 was the v3 operational smoke-test option; Q4 and Q9 remained development-exposed cases. The earlier v2 instructions used Q4 for debugging. Optional live Cellar verification was separate and was not required for the frozen-corpus evaluation.

Per-question writes use exclusive creation and do not overwrite completed official files. This is not blanket write protection for other export or summary cells. Any separately authorized new experiment must use separate output paths and must not replace the frozen evaluation.

**Historical v1/v2 records:** the original Q4 records and manifests remain under `archive/diagnostics/data/build04_nb09_runs/` and `archive/diagnostics/data/build04_nb09_v2_runs/`. They are not the v3 output directory.


In [21]:
# Research budget and manifest gate, required before any paid question runs.
assert PARITY_OK and matched==50, "Retrieval parity must be 50/50"
assert all(qid in questions for qid in SELECT_QIDS)
RUN_DIR.mkdir(parents=True,exist_ok=True)
RUN_SETTINGS={
    "experiment_id":EXPERIMENT_ID,"benchmark_sha256":benchmark_hash,
    "model":MODEL_NAME,"reasoning_effort":REASONING_EFFORT,
    "chunks_sha256":hashlib.sha256(FILES["chunks"].read_bytes()).hexdigest(),
    "retrieval_config_sha256":hashlib.sha256(FILES["retrieval_config"].read_bytes()).hexdigest(),
    "initial_max":MAX_INITIAL_SEARCHES,"targeted_max":MAX_TARGETED_SEARCHES,
    "validation_max":MAX_VALIDATION_ROUNDS,"max_llm_calls":MAX_LLM_CALLS,
    "enable_cellar_live":ENABLE_CELLAR_LIVE,
    "coverage_tracker":"planner_task_plus_subissue_v1",
    "gap_classification":"type_aware_v1",
    "runtime_version":"nb09_build04_v3",
}
manifest_path=RUN_DIR/"manifest.json"
if RUN_OFFICIAL:
    if manifest_path.exists():
        previous=json.loads(manifest_path.read_text(encoding="utf-8"))
        assert previous==RUN_SETTINGS,"Experiment configuration changed. Use a new EXPERIMENT_ID / directory."
    else:
        with manifest_path.open("x",encoding="utf-8") as f:
            json.dump(RUN_SETTINGS,f,indent=2,ensure_ascii=False)
else:
    print("DRY RUN: manifest not written; you may adjust settings before the first official run.")
print("Parity",matched,"/50 | output",RUN_DIR,"| paid runs",RUN_OFFICIAL,"| selected",SELECT_QIDS)

Parity 50 /50 | output data\build04_nb09_v3_runs | paid runs True | selected ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10']


In [22]:
def execute_one_qid(qid:str)->dict|None:
    assert PARITY_OK
    assert qid in questions
    outpath=RUN_DIR/f"{qid}.json"
    if outpath.exists():
        print("SKIP: previous result is preserved:",outpath)
        return None
    start=datetime.now(timezone.utc)
    t0=time.perf_counter()
    LLM_USAGE.update({"input_tokens":0,"output_tokens":0,"calls":0})
    state=new_state(questions[qid],qid,case_context={"benchmark_case":"Türkiye-origin dried figs; aflatoxin B1 detected; measured concentration and consignment status not provided"})
    try:
        answer=agent_graph.invoke(state,config={"recursion_limit":20})
        assert answer.get("final_answer"),"Empty final report"
        record={
            "experiment_id":EXPERIMENT_ID,"build":"Build04","qid":qid,
            "question":questions[qid],"benchmark_id":benchmark["benchmark_id"],
            "benchmark_sha256":benchmark_hash,"settings":RUN_SETTINGS,
            "model":MODEL_NAME,"reasoning_effort":REASONING_EFFORT,
            "plan":answer.get("investigation_plan",[]),
            "evidence":answer.get("evidence",[]),"claims":answer.get("claims",[]),
            "gaps":answer.get("gaps",[]),"conflicts":answer.get("conflicts",[]),
            "coverage_items":answer.get("coverage_items",[]),
            "coverage_summary":answer.get("coverage_summary",{}),
            "classified_gaps":answer.get("classified_gaps",[]),
            "current_law_calls":answer.get("current_law_checks",[]),
            "trace":answer.get("trace",[]),"final_output":answer["final_answer"],
            "source_facts":answer.get("source_facts",[]),
            "audit_resolution":answer.get("audit_resolution",""),
            "draft_output":answer.get("original_draft") or answer.get("draft_answer",""),
            "search_calls":answer.get("search_calls",0),
            "initial_searches":answer.get("initial_searches",0),
            "targeted_searches":answer.get("targeted_searches",0),
            "validation_rounds":answer.get("validation_rounds",0),
            "llm_calls":LLM_USAGE["calls"],"usage":answer.get("usage",{}),
            "audit_issues":answer.get("audit_issues",[]),
            "stop_reason":("coverage_complete" if not answer.get("coverage_summary",{}).get("unresolved_critical",0)
               else "validation_budget" if answer.get("validation_rounds",0)>=MAX_VALIDATION_ROUNDS
               else "search_budget" if answer.get("search_calls",0)>=MAX_TOTAL_SEARCHES
               else "unsearchable_or_unresolved_gaps"),
            "targeted_search_yield":sum(len(t.get("added_ids",[])) for t in answer.get("trace",[])
                if t.get("node")=="retriever" and t.get("phase")=="targeted"),
            "started_utc":start.isoformat(),"completed_utc":datetime.now(timezone.utc).isoformat(),
            "latency_seconds":round(time.perf_counter()-t0,2),
            "status":"completed_pending_human_review",
        }
        with outpath.open("x",encoding="utf-8") as f:
            json.dump(record,f,ensure_ascii=False,indent=2)
        print("SAVED",qid,"| searches",record["search_calls"],
              "| rounds",record["validation_rounds"],"| LLM calls",record["llm_calls"],
              "| tokens",record["usage"])
        return record
    except Exception as exc:
        err={"qid":qid,"error_type":type(exc).__name__,"error_message":str(exc),
             "failed_utc":datetime.now(timezone.utc).isoformat(),"llm_usage":dict(LLM_USAGE)}
        error_path=RUN_DIR/f"{qid}_error.json"
        if not error_path.exists():
            with error_path.open("x",encoding="utf-8") as ef:
                json.dump(err,ef,ensure_ascii=False,indent=2)
        else: print("Existing error log preserved:",error_path)
        print("FAILED",qid,err["error_type"],err["error_message"][:350])
        raise

if RUN_OFFICIAL:
    for qid in SELECT_QIDS:
        execute_one_qid(qid)
else:
    print("Official run disabled. No OpenAI requests made.")

SAVED Q1 | searches 7 | rounds 2 | LLM calls 7 | tokens {'input_tokens': 45685, 'output_tokens': 11420}
SAVED Q2 | searches 3 | rounds 2 | LLM calls 7 | tokens {'input_tokens': 20819, 'output_tokens': 6964}
SAVED Q3 | searches 4 | rounds 1 | LLM calls 5 | tokens {'input_tokens': 17684, 'output_tokens': 6146}
SAVED Q4 | searches 5 | rounds 1 | LLM calls 6 | tokens {'input_tokens': 24674, 'output_tokens': 9440}
SAVED Q5 | searches 8 | rounds 2 | LLM calls 8 | tokens {'input_tokens': 57141, 'output_tokens': 15496}
SAVED Q6 | searches 7 | rounds 2 | LLM calls 8 | tokens {'input_tokens': 51916, 'output_tokens': 17374}
SAVED Q7 | searches 8 | rounds 2 | LLM calls 7 | tokens {'input_tokens': 42549, 'output_tokens': 13161}
SAVED Q8 | searches 6 | rounds 2 | LLM calls 7 | tokens {'input_tokens': 45885, 'output_tokens': 12448}
SAVED Q9 | searches 8 | rounds 2 | LLM calls 7 | tokens {'input_tokens': 55002, 'output_tokens': 15332}
SAVED Q10 | searches 8 | rounds 2 | LLM calls 7 | tokens {'input_to

## 15. Saved-result comparison and human evaluation worksheet

Build01/02 answer text is read from the frozen NB04b CSV under `results/build01_02/`. Build03-B NB08 outputs are optional comparison records, not inputs required to execute the Build04 workflow. **Correct/Grounded/Complete remain human-assigned** against the frozen Verified Reference after the run; never expose those grades to the Agent.

The NB08 files use the `Q#.json` naming convention under `results/build03/`, which is the current `BUILD03_DIR`. This section also contains CSV export and worksheet-creation logic; it is not entirely read-only. No automatic grades are generated, and the frozen evaluation records must be preserved.


In [ ]:
BUILD03_DIR=__import__("pathlib").Path("results/build03")
rows=[]
for qid in questions:
    p=RUN_DIR/f"{qid}.json"
    obj=json.loads(p.read_text(encoding="utf-8")) if p.exists() else None
    base03=BUILD03_DIR/f"{qid}.json"
    old=json.loads(base03.read_text(encoding="utf-8")) if base03.exists() else None
    rows.append({
        "qid":qid,
        "Build02_frozen_answer":comparison_df.loc[comparison_df.qid==qid,"rag_answer"].iloc[0][:250],
        "Build03_answer_available":bool(old),
        "Build04_status":obj["status"] if obj else "NOT_RUN",
        "Build04_searches":obj.get("search_calls") if obj else None,
        "Build04_targeted":obj.get("targeted_searches") if obj else None,
        "Build04_supported_claims":sum(c.get("status")=="supported" for c in obj["claims"]) if obj else None,
        "Build04_audit_issues":len(obj.get("audit_issues",[])) if obj else None,
        "Build04_audit_resolution":obj.get("audit_resolution") if obj else None,
        "Build04_source_facts":len(obj.get("source_facts",[])) if obj else None,
    })
summary_df=pd.DataFrame(rows)
display(summary_df)
# Blank worksheet: grading is done by a human against fixed Notion reference.
EVAL_PATH=RUN_DIR/"build04_q1_q10_human_evaluation.csv"
if not EVAL_PATH.exists():
    pd.DataFrame([{
        "qid":qid,"correct":"","grounded":"","complete":"",
        "verified_reference_checked":"","critical_legal_error":"",
        "failure_category":"","notes":""
    } for qid in questions]).to_csv(EVAL_PATH,index=False,encoding="utf-8-sig")
print("Evaluation worksheet:",EVAL_PATH)

## 16. Final evaluation interpretation and freeze protocol

- **Preservation**: v1 and v2 official results remain untouched. v3 has its own manifest, parity audit and official runs folder.
- **Evidence correctness**: human Correct/Grounded/Complete scores use the already-frozen frozen reference outside runtime. Internal coverage is a *diagnostic*, not a replacement for human completeness.
- **Development exposure**: Q4 and Q9 were used to inform v3 architecture. Report them separately from eight less-exposed questions; do not claim independent generalization from their scores.
- **Source reliability**: Cellar metadata is not article-level validity checking; with live disabled, frozen document findings are labeled as snapshot-only.
- **Freeze**: save notebook hash and results. After Q1–Q10 official runs, no tuning or silent reruns to improve the main evaluation. Bugs are documented as new supplementary experiments, not replacements.
- **Metrics**: task/subissue critical coverage ratio, unresolved gaps by type, targeted-search yield, audit resolution, LLM input/output tokens, latency and human rubric.
